# 37 Reference models: evaluation and comparison (CPU runtime)

Scores the predictions of `36_reference_models` exactly as our own models are scored (EX, step-based R-VES, cost relative to gold), then prints one table with our models and the reference models on dev and mini-dev:

1. against **our RL model kept on validation**: is a reference model significantly more accurate (McNemar), and on questions both answer correctly, whose queries are cheaper (sign test)?
2. against **our untuned base model** (zero-shot), the common starting point.

Last, the average answer length per model: the reasoning models write many more tokens per question, which is inference cost.

**Runtime > Change runtime type > CPU**.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Run order: the two published 7B text-to-SQL models first, then the 32B model, then our base model with
# OmniSQL's prompt. Keys of configs/reference.yaml; each writes runs/ref_<key>.
MODELS = ["omnisql7b", "arctic7b", "qwen32b", "qwen7b_omnisql"]
R = f"{DRIVE_ROOT}/effsql/runs"
import os
import json
CFG = {m: json.load(open(f"{R}/ref_{m}/predict_stats_dev.json")) for m in MODELS
       if os.path.exists(f"{R}/ref_{m}/predict_stats_dev.json")}
print("predicted:", list(CFG), "| missing:", [m for m in MODELS if m not in CFG])

### 1. Stage the dev databases

In [ ]:
!effsql stage --split dev

### 2. Score the reference models (resumable)

In [ ]:
for m in CFG:
    model = json.loads(open(f"{R}/ref_{m}/predictions_dev.jsonl").readline())["model"]
    method = "reference-" + CFG[m]["style"]
    !effsql evaluate --pred {R}/ref_{m}/predictions_dev.jsonl --split dev --timing none --run ref_{m}_dev --model {model} --method {method}
    !effsql evaluate --pred {R}/ref_{m}/predictions_minidev.jsonl --split minidev --timing quick --run ref_{m}_minidev --model {model} --method {method}

### 3. Everything against our RL model kept on validation

Columns as in the earlier reports: EX, R-VES (step-based, gold = 100, wrong = 0) with a 95% bootstrap interval, R-VES/EX (the efficiency part alone), answers at least 2x cheaper or slower than gold, geometric mean cost relative to gold. Below each table: each model against our RL model on the same questions (R-VES and EX differences with p-values; cheaper / costlier when both are correct, by at least 10%). Saved as `reference_report/report.txt`.

In [ ]:
OURS = ["pool_v1_{split}_base", "sft_v1_gold_e1", "sft_v1_best_e1", "rl_v1_it8", "rl_v1"]
REFS = [f"ref_{m}" for m in MODELS]
LABELS = ",".join(["pool_v1_{split}_base=ours: zero-shot 7B", "sft_v1_gold_e1=ours: SFT gold",
                   "sft_v1_best_e1=ours: SFT best", "rl_v1_it8=ours: RL (kept)", "rl_v1=ours: RL 24 it",
                   "ref_omnisql7b=OmniSQL-7B", "ref_arctic7b=Arctic-R1-7B", "ref_qwen32b=Qwen-32B zero-shot",
                   "ref_qwen7b_omnisql=Qwen-7B OmniSQL prompt"])
!python -m effsql.train report --runs "{','.join(OURS + REFS)}" --ref rl_v1_it8 --splits dev,minidev --labels "{LABELS}" --out reference_report

### 4. Everything against our untuned base model (zero-shot)

In [ ]:
OURS = ["pool_v1_{split}_base", "sft_v1_gold_e1", "sft_v1_best_e1", "rl_v1_it8", "rl_v1"]
REFS = [f"ref_{m}" for m in MODELS]
LABELS = ",".join(["pool_v1_{split}_base=ours: zero-shot 7B", "sft_v1_gold_e1=ours: SFT gold",
                   "sft_v1_best_e1=ours: SFT best", "rl_v1_it8=ours: RL (kept)", "rl_v1=ours: RL 24 it",
                   "ref_omnisql7b=OmniSQL-7B", "ref_arctic7b=Arctic-R1-7B", "ref_qwen32b=Qwen-32B zero-shot",
                   "ref_qwen7b_omnisql=Qwen-7B OmniSQL prompt"])
BASE = OURS[0]  # the zero-shot runs: pool_v1_dev_base, pool_v1_minidev_base
!python -m effsql.train report --runs "{','.join(OURS + REFS)}" --ref "{BASE}" --splits dev,minidev --labels "{LABELS}" --out reference_report_vs_base

### 5. Answer length (tokens per answer on dev)

In [ ]:
def mean_tokens(path):
    last = {}
    for line in open(path):
        r = json.loads(line)
        last[r["key"]] = r["completions"][0]["n_tokens"] if r["completions"] else 0
    return sum(last.values()) / max(len(last), 1)

RAW = {"ours: zero-shot 7B": f"{R}/pool_v1/predict_raw_dev_base.jsonl",
       "ours: RL (kept)": f"{R}/rl_v1_it8/predict_raw_dev.jsonl"}
RAW.update({m: f"{R}/ref_{m}/predict_raw_dev.jsonl" for m in CFG})
for name, f in RAW.items():
    if os.path.exists(f):
        print(f"{name:<22}{mean_tokens(f):>8.0f}")

### Done: disconnect the runtime